# Clonamos el repositorio con los modelos y herramientas¶

In [1]:
!git clone https://github.com/dannasalazar11/Msc_thesis.git

Cloning into 'Msc_thesis'...
remote: Enumerating objects: 450, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 450 (delta 13), reused 0 (delta 0), pack-reused 427 (from 1)
Receiving objects: 100% (450/450), 49.97 MiB | 41.37 MiB/s, done.
Resolving deltas: 100% (290/290), done.


In [2]:
import sys
sys.path.append('/kaggle/working/Msc_thesis')

from gmrrnet_adhd.utils import get_segmented_data, train_L24O_cv
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy('mixed_float16')


import tensorflow as tf
import numpy as np
import random
import os

# Establecer semilla
seed = 42

# Semillas para módulos principales
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)

2025-06-30 23:48:00.514596: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751327281.004823      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751327281.149415      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# Importar base de datos segmentada (Segmentos de 4 seg con translape del 50%, es decir, de 2 seg)

In [3]:
from gmrrnet_adhd.models.spatio_temporal import prepare_streams_4s

X, y, sbjs = get_segmented_data()
X.shape, y.shape, len(sbjs)

((8213, 19, 512), (8213, 2), 8213)

## Preprocesamiento de los datos mencionado por la propuesta

| Variable   | Forma resultante | Cálculo exacto                                                                                                                                                                                                                |
| ---------- | ---------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **`freq`** | `(N, 20, 1)`     | - PSD con `welch(signal, fs=128, nperseg=512)`.<br>- Potencia media en **20 bandas log‑espaciadas** entre 1 Hz y 64 Hz.<br>- Promedio sobre canales → vector de 20.<br>- Se añade un eje final de tamaño 1.                   |
| **`temp`** | `(N, 10, 1)`     | - Se recortan 510 muestras (de 512).<br>- Se dividen en **10 ventanas** consecutivas de 51 muestras (≈ 400 ms).<br>- **Media aritmética** dentro de cada ventana promediando canales.<br>- Se añade un eje final de tamaño 1. |
| **`spat`** | `(N, C, 1)`      | - Para cada canal: **RMS** del segmento `sqrt(mean(x**2))`.<br>- Se añade un eje final de tamaño 1.                                                                                                                           |

In [4]:
freq, temp, spat = prepare_streams_4s(X, fs=128)

freq.shape, temp.shape, spat.shape

((8213, 20, 1), (8213, 10, 1), (8213, 19, 1))

In [5]:
from sklearn.preprocessing import StandardScaler

scaler_f = StandardScaler().fit(freq.reshape(-1, 20))
scaler_t = StandardScaler().fit(temp.reshape(-1, 10))
scaler_s = StandardScaler().fit(spat.reshape(-1, spat.shape[1]))

freq = scaler_f.transform(freq.reshape(-1, 20)).reshape(freq.shape)
temp = scaler_t.transform(temp.reshape(-1, 10)).reshape(temp.shape)
spat = scaler_s.transform(spat.reshape(-1, spat.shape[1])).reshape(spat.shape)

# Importamos el modelo y definimos los hiperparámetros

In [6]:
from gmrrnet_adhd.models.spatio_temporal import build_eeg_attention_model
from tensorflow.keras.optimizers import Adam

model_name="spatio_temporal"

model_args =    {'freq_shape' : freq.shape[1:],   # (20,1)
                 'temp_shape' : temp.shape[1:],   # (10,1)
                 'spat_shape' : spat.shape[1:]}   # (19,1)

compile_args = {'optimizer':lambda: Adam(1e-4, clipnorm=1.0),
    "loss": "categorical_crossentropy",
    "metrics": ["accuracy"]
}

model = build_eeg_attention_model(
    **model_args
)

model.summary()

I0000 00:00:1751327401.132839      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1751327401.133598      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "EEG_Attention_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ freq_input (InputLayer)   │ (None, 20, 1)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ temp_input (InputLayer)   │ (None, 10, 1)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ spat_input (InputLayer)   │ (None, 19, 1)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast (Cast)               │ (None, 20, 1)          │              0 │ freq_input[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_2 (Cast)             │ (None, 10, 1)          │              0 │ temp_input[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_4 (Cast)             │ (None, 19, 1)          │              0 │ spat_input[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 20, 64)         │            128 │ cast[0][0]             │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_6 (Dense)           │ (None, 10, 64)         │            128 │ cast_2[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_12 (Dense)          │ (None, 19, 64)         │            128 │ cast_4[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ positional_encoding       │ (None, 20, 64)         │              0 │ dense[0][0]            │
│ (PositionalEncoding)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ positional_encoding_1     │ (None, 10, 64)         │              0 │ dense_6[0][0]          │
│ (PositionalEncoding)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ positional_encoding_2     │ (None, 19, 64)         │              0 │ dense_12[0][0]         │
│ (PositionalEncoding)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_1 (Cast)             │ (None, 20, 64)         │              0 │ positional_encoding[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_3 (Cast)             │ (None, 10, 64)         │              0 │ positional_encoding_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_5 (Cast)             │ (None, 19, 64)         │              0 │ positional_encoding_2… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ transformer_encoder_block │ (None, 20, 64)         │         83,200 │ cast_1[0][0]           │
│ (TransformerEncoderBlock) │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ transformer_encoder_bloc… │ (None, 10, 64)         │         83,200 │ cast_3[0][0]           │
│ (TransformerEncoderBlock) │                        │                │                        │
├──────────────────────

 Total params: 574,082 (2.19 MB)

 Trainable params: 574,082 (2.19 MB)

 Non-trainable params: 0 (0.00 B)

# Resultados - Leave 24 Subjects Out

In [7]:
import os

import pickle

with open("/kaggle/input/ieee-tdah-control-database/folds.pkl", "rb") as f:
    folds = pickle.load(f)

In [8]:
# X_total is a list: [freq, temp, spat]


In [9]:
import numpy as np

X_total = [freq, temp, spat]            # each array shape (N, …)

results = {}

for i in range(6):
    result = train_L24O_cv(build_eeg_attention_model, X_total, y, sbjs, model_args, compile_args, folds, model_name='spatio_temporal')
    results[i] = result

Fold 1/5. Test subjects: ['v28p', 'v274', 'v1p', 'v231', 'v22p', 'v29p', 'v206', 'v238', 'v31p', 'v35p', 'v177', 'v200', 'v112', 'v113', 'v48p', 'v140', 'v131', 'v125', 'v55p', 'v143', 'v43p', 'v305', 'v134', 'v114']


I0000 00:00:1751327438.462600      71 service.cc:148] XLA service 0x7df978005270 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1751327438.463882      71 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1751327438.463901      71 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1751327443.153598      71 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1751327461.489919      71 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 98ms/step
Fold metrics: {'accuracy': 0.7227179135209334, 'recall': 0.7178697116619088, 'precision': 0.7303515678023619, 'kappa': 0.43951506933838125, 'auc': 0.7178697116619089}
Fold 2/5. Test subjects: ['v18p', 'v39p', 'v234', 'v32p', 'v190', 'v6p', 'v254', 'v204', 'v24p', 'v183', 'v246', 'v219', 'v298', 'v41p', 'v47p', 'v308', 'v52p', 'v300', 'v59p', 'v299', 'v302', 'v51p', 'v109', 'v127']
53/53 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step
Fold metrics: {'accuracy': 0.6241007194244604, 'recall': 0.5898947310124507, 'precision': 0.6409877209183255, 'kappa': 0.1908801138825278, 'auc': 0.5898947310124507}
Fold 3/5. Test subjects: ['v215', 'v3p', 'v209', 'v37p', 'v213', 'v15p', 'v284', 'v181', 'v19p', 'v34p', 'v263', 'v244', 'v138', 'v121', 'v46p', 'v54p', 'v120', 'v310', 'v147', 'v50p', 'v56p', 'v107', 'v297', 'v108']
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step
Fold metrics: {'accuracy': 0.8235294117647058, 'recall': 0.8159844678849503, 'precision': 0.8159844678849503, 'ka

In [10]:
for i in range(6):
    result = results[i]
    accs = []
    for r in result:
        accs.append(r['accuracy'])
    
    print(i, '->', np.mean(accs))

0 -> 0.7469783440515767
1 -> 0.7393795787624513
2 -> 0.7339310515586597
3 -> 0.7477994429441153
4 -> 0.7420178127549678
5 -> 0.7330924796879879


In [11]:
import pickle

with open(f'results_L24SO_{model_name}.pkl', 'wb') as f:
    pickle.dump(results, f)